In [57]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go


In [58]:
# Load the data
df = pd.read_csv('data/apple_sales_data_transformed.csv')

In [59]:
df.head()

,sale_id,sale_date,year,quarter,month,country,region,city,product_name,category,...,revenue_local_currency,sales_channel,payment_method,customer_segment,customer_age_group,previous_device_os,customer_rating,return_status,month_calender,revenue_no_discount_usd
0,APPL-00000001,2022-01-03,2022,Q1,January,Argentina,South America,Buenos Aires,AirPods (3rd Gen),AirPods,...,134344.84,Third-Party Retailer,Cash,Government,45–54,NaN,4.1,Kept,2022-01-01,159.27
1,APPL-00000002,2022-01-04,2022,Q1,January,Argentina,South America,Buenos Aires,USB-C Woven Charge Cable,Accessories,...,115597.15,Authorized Reseller,Debit Card,Business,45–54,NaN,4.8,Kept,2022-01-01,149.95
2,APPL-00000003,2022-05-18,2022,Q2,May,Argentina,South America,Buenos Aires,Apple Watch Series 8,Apple Watch,...,1066341.76,Corporate / B2B,Credit Card,Individual,18–24,NaN,4.3,Kept,2022-05-01,1175.68
3,APPL-00000004,2022-05-23,2022,Q2,May,Argentina,South America,Buenos Aires,MacBook Pro 14-inch (M3),Mac,...,3506044.78,Carrier Store,Credit Card,Education,45–54,NaN,NaN,Kept,2022-05-01,3865.54
4,APPL-00000005,2022-07-13,2022,Q3,July,Argentina,South America,Buenos Aires,Apple Watch Ultra 2,Apple Watch,...,1952780.07,Apple Store,Net Banking,Education,18–24,NaN,NaN,Kept,2022-07-01,2266.32


In [60]:
df_net = df[df['return_status'] != 'Returned']

# Executive overview

In [61]:
total_revenue = df_net['revenue_usd'].sum()
total_units = df_net['units_sold'].sum()
avg_discount = df_net['discount_pct'].mean()

kpis = pd.DataFrame({
    "Metric": ["Total Revenue", "Units Sold", "Average Discount"],
    "Value": [
        f"${total_revenue/1e6:.1f}M",
        f"{total_units:,}",
        f"{avg_discount:.1f}%"
    ]
})

ind1 = go.Figure(go.Indicator(
    mode="number",
    value=total_revenue,
    number={'prefix': "$", 'valueformat': ",.0f"},
    title={"text": "Total Revenue between 2022 and 2025"}
))

ind2 = go.Figure(go.Indicator(
    mode="number",
    value=total_units,
    number={'valueformat': ".0f"},
    title={"text": "Total Units Sold between 2022 and 2025"}
))

ind3 = go.Figure(go.Indicator(
    mode="number",
    value=avg_discount,
    number={'suffix': "%", 'valueformat': ".1f"},
    title={"text": "Average Discount Percentage between 2022 and 2025"}
))

ind1.show()
ind2.show()
ind3.show()

In [62]:
# Line chart of revenue over time

df_fig1 = df_net.groupby('month_calender')['revenue_usd'].sum().reset_index()
fig1 = px.line(df_fig1, x = 'month_calender', 
               y = 'revenue_usd', 
               markers=True,
               labels = {'month_calender': '', 'revenue_usd': 'Revenue (USD)'},
               template='simple_white')

fig1.update_yaxes(
    tickprefix='$', ticks='outside'
)

fig1.update_xaxes(
    hoverformat='%b %Y'
)

fig1.update_traces(
    mode='markers+lines', hovertemplate=None
)

fig1.update_layout(
    font_family='rockwell',
    hovermode='x unified',
    title = {
        'text': 'Apple Sales Revenue Over Time <br> <sup style="font-size:0.8em;color:gray;">Hover over data points to see specific USD values</sup>',
        'x': 0.5
    }
)

fig1.add_shape(
    type='line', line_color='salmon', line_width=3, opacity=1, line_dash='dot',
    x0=df_fig1['month_calender'].min(), x1=df_fig1['month_calender'].max(), 
    y0=df_fig1['revenue_usd'].mean(), y1=df_fig1['revenue_usd'].mean(),

)

fig1.add_annotation(
    x=df_fig1['month_calender'].max(), y=df_fig1['revenue_usd'].mean(),
    text='Average Revenue', showarrow=False, yshift=-10, font_color='salmon'
)

fig1.update_traces(mode='markers+lines', hovertemplate=None)


fig1.show()

In [63]:
# Bar chart for sales per year
df_fig2 = df_net.groupby(df['year'])['revenue_usd'].sum().reset_index()
df_fig2['year'] = df_fig2['year'].astype(str)
df_fig2['revenue_millions'] = df_fig2['revenue_usd'] / 1e6


fig2 = px.bar(
    df_fig2,
    x = 'year',
    y = 'revenue_usd',
    labels = {'year':'Year', 'revenue_usd':'Revenue (USD)'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig2['revenue_millions'].map('${:.1f}M'.format)
)

fig2.update_layout(
    font_family = 'rockwell',
        title={
        "text": "Total Revenue by Year<br><sup style='font-size:12px; color:gray;'>Hover over bars to see specific USD values</sup>",
        "x": 0.5
    }
)

fig2.update_traces(
    hovertemplate='$%{y:,.2f} USD<extra></extra>',
)

fig2.update_yaxes(
    tickprefix='$', ticks='outside'
)


fig2.show()

In [64]:
# Bar chart for units sold per year
df_fig3 = df_net.groupby('year')['units_sold'].sum().reset_index()
df_fig3['year'] = df_fig3['year'].astype(str)  

fig3 = px.bar(
    df_fig3,
    x = 'year',
    y = 'units_sold',
    labels = {'year':'Year', 'units_sold':'Units Sold'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig3['units_sold'].map('{:.0f}'.format)
)

fig3.update_layout(
    font_family = 'rockwell',
    title = {
        'text': 'Total Units Sold by Year',
        'x': 0.5
    }
)

fig3.update_yaxes(
    ticks='outside',
    tickformat='~s'
)

fig3.update_traces(
    hovertemplate='%{y:.0f} Units<extra></extra>'
)

fig3.show()

In [65]:
# Bar chart for average discount percentage per year
df_fig4 = df_net.groupby('year')['discount_pct'].mean().reset_index()
df_fig4['year'] = df_fig4['year'].astype(str)

fig4 = px.bar(
    df_fig4,
    x = 'year',
    y = 'discount_pct',
    labels = {'year':'Year', 'discount_pct':'Average Discount (%)'},
    template='simple_white',
    color_discrete_sequence=['steelblue'],
    text=df_fig4['discount_pct'].map('{:.1f}%'.format)
)

fig4.update_layout(
    font_family = 'rockwell',
    title = {
        'text': 'Average Discount Percentage by Year',
        'x': 0.5
    }
)

fig4.update_yaxes(
    ticks='outside',
    tickformat='.1f%',
    range=[0, df_fig4['discount_pct'].max() * 1.2]
)

fig4.update_traces(
    hovertemplate='%{y:.1f}%<extra></extra>'
)

fig4.show()

# Geographic performance

In [66]:
# Pie chart for revenue per region
df_pie = df_net.groupby('region')[['revenue_usd', 'units_sold']].sum().reset_index()
pie1 = px.pie(
    df_pie, 
    values='revenue_usd', 
    names='region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)
              
        

pie1.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Revenue by Region <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific USD values</sup>',
        'x': 0.5,
    },
    legend_title_text='Region'
)

pie1.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='$%{value:,.2f} USD<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)



pie1.show()

In [67]:
# Choropleth map for revenue by country

# note to self: give user option to swap between metrics

df_map = df_net.groupby(['country', 'region'])['revenue_usd'].sum().reset_index()


fig_map = go.Figure(data=go.Choropleth(
    locations=df_map['country'],
    z = df_map['revenue_usd'],
    locationmode='country names',
    text=df_map['country'],
    colorscale='Blues',
    marker_line_color='black',
    marker_line_width=0.5,
    colorbar_ticksuffix='$',
    colorbar_title='Revenue (USD)'
))

fig_map.update_layout(
    title= {
        'text': 'Revenue by Country (2022-2025) <br> <sup style="font-size:12px;color:gray">Hover over countries to see specific USD values</sup>',
        'x': 0.5
    },
    geo=dict(
        showframe=False,
        showcoastlines=True,
        projection_type='equirectangular'
    ),
    annotations=[
        dict(
            x=0.5,
            y=-0.1,
            xref='paper',
            yref='paper',
            text='* Note: Dataset only contains data for 47 countries, so some regions may be underrepresented.',
            showarrow=False,
            font=dict(size=10, color='gray')
        )
    ],
    font_family='rockwell')

fig_map.update_traces(
    hovertemplate='%{text}: $%{z:,.2f} USD<extra></extra>'
)


fig_map.show()

In [68]:
pie2 = px.pie(
    df_pie, 
    values='units_sold', 
    names='region',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)

pie2.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Units Sold by Region <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific unit values</sup>',
        'x': 0.5,
    },
    legend_title_text='Region'
)

pie2.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='%{value:.0f} Units<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)

pie2.show()

In [69]:
# Table for top 10 countries by average revenue per sale

# Note to self: give users option to change table between revenue and units sold

top10_country_revenue = df_net.groupby('country')['revenue_usd'].mean().reset_index().sort_values(by='revenue_usd', ascending=False)

top10_countries = go.Figure(data=[go.Table(
    columnorder = [0, 1, 2],
    columnwidth = [40, 150, 150],
    header=dict(values=['Rank','Country', 'Revenue (USD)'],
                fill_color='steelblue',
                font=dict(color='white', 
                          size=12,
                          family='rockwell'),),
    cells=dict(values=[list(range(1, len(top10_country_revenue) + 1)), top10_country_revenue['country'], top10_country_revenue['revenue_usd'].map('${:,.2f}'.format)],
                fill_color="#E5E6E6",
                font = dict(color='black',
                            size = 11,
                            family='rockwell')),)      
    ]
)
                          
top10_countries.update_layout(
    title = {
        'text': 'Top Countries by Average Revenue per Sale (2022-2025) <br> <sup style="font-size:12px;color:gray">Scroll table to explore more countries</sup>',
        'font': {
            'family': 'rockwell',
            'size': 16
        },
        'x': 0.5
    }
)

top10_countries.show()


# Product & Pricing Analysis

In [70]:
# Pie chart for revenue per region

# Note to self: give users option to change pie chart between revenue and units sold

df_pie2 = df_net.groupby('category')['revenue_usd'].sum().reset_index()
pie_product = px.pie(
    df_pie2, 
    values='revenue_usd', 
    names='category',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white'
)
              
        

pie_product.update_layout(
    font_family='rockwell',
    title = {
        'text': 'Percentage of Revenue by Product Category <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific USD values</sup>',
        'x': 0.5,
    },
    legend_title_text='Product Category',
)

pie_product.update_traces(
    textposition='inside', 
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='$%{value:,.2f} USD<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)



pie_product.show()

In [71]:
# Table for revenue by product name

### Note for self: Give dashboard option to choose metric and product category.

df_prod_rev = df_net.groupby('product_name')[['revenue_usd', 'units_sold']].sum().reset_index().sort_values(by='revenue_usd', ascending=False)

top10_products = go.Figure(data=[go.Table(
    columnorder = [0, 1, 2],
    columnwidth = [40, 250, 150],
    header=dict(values=['Rank','Product Name', 'Revenue (USD)'],
                fill_color='steelblue',
                font=dict(color='white', 
                          size=12,
                          family='rockwell'),),
    cells=dict(values=[list(range(1, len(df_prod_rev) + 1)), df_prod_rev['product_name'], df_prod_rev['revenue_usd'].map('${:,.2f}'.format)],
                fill_color="#E5E6E6",
                font = dict(color='black',
                            size = 11,
                            family='rockwell')),)      
    ]
)

top10_products.update_layout(
    title = {
        'text': 'Revenue by Product Name (2022-2025) <br> <sup style="font-size:12px;color:gray">Scroll table to explore more products</sup>',
        'font': {
            'family': 'rockwell',
            'size': 16
        }
    },
    title_x=0.5
)

top10_products.show()

In [72]:
df_disc_count = (
    df_net
    .groupby(["category", "discount_pct"])
    .size()
    .reset_index(name="n_discounts")
)

# Calculate total sales per category
total_sales_per_category = df_net.groupby('category').size().reset_index(name='total_sales')

# Merge to get the denominator for each category
df_disc_count = df_disc_count.merge(total_sales_per_category, on='category')

# Calculate percentage of total sales per category
df_disc_count['pct_of_category_sales'] = (df_disc_count['n_discounts'] / df_disc_count['total_sales']) * 100

df_disc_count['discount_pct'] = df_disc_count['discount_pct'].astype(str)

category_order = (
    df_disc_count.groupby('category')['pct_of_category_sales']
    .sum()
    .sort_values(ascending=False)
    .index
)

fig_disc_count = px.bar(
    df_disc_count,
    x='category',
    y='pct_of_category_sales',
    color='discount_pct',
    custom_data=['discount_pct'],
    color_discrete_sequence=px.colors.sequential.Blues,
    labels={
        'category':'Product Category',
        'pct_of_category_sales':'Percentage of Category Sales (%)',
        'discount_pct':'Discount Percentage'
    },
    template='simple_white',
    category_orders={'category': category_order}
)

fig_disc_count.update_layout(
    font_family='rockwell',
    title={
        'text': 'Percentage of Category Sales by Discount Percentage (2022-2025)'
                '<br><sup style="font-size:12px;color:gray">Hover over bars to see specific discount percentages</sup>',
        'x':0.5
    },
    legend_traceorder='reversed'
)

fig_disc_count.update_traces(
    hovertemplate="%{customdata[0]}%<br>%{y:.2f}% of category sales<extra></extra>",
    marker_line_color='steelblue',
)

fig_disc_count.show()

In [73]:
# pareto chart for revenue by product name

# note to self: give users option to filter for category

df_pareto = df_net[df_net['category'] == 'Accessories'].groupby('product_name')['revenue_usd'].sum().reset_index().sort_values(by='revenue_usd', ascending=False)

fig_pareto = go.Figure()

fig_pareto.add_trace(go.Bar(
    x = df_pareto['product_name'],
    y = df_pareto['revenue_usd'],
    name='Revenue',
    marker_color='steelblue',
))

fig_pareto.add_trace(go.Scatter(
    x = df_pareto['product_name'],
    y = (df_pareto['revenue_usd'].cumsum() / df_pareto['revenue_usd'].sum() * 100).round(1),
    name='Cumulative Percentage',
    mode='lines+markers',
    marker_color='salmon',
    yaxis='y2'
))

fig_pareto.update_layout(
    title = {
        'text': 'Pareto Chart of Revenue by Product Name for Accessories (2022-2025) <br> <sup style="font-size:12px;color:gray">Hover over bars and points to see specific values</sup>',
        'x': 0.5
    },
    yaxis=dict(
        title='Revenue (USD)',
        showgrid=False
    ),
    yaxis2=dict(
        title='Cumulative Percentage (%)',
        overlaying='y',
        side='right',
        showgrid=False,
        ticksuffix='%',
        range=[0, 115]

    ),
    xaxis=dict(
        tickangle=-45
    ),
    legend=dict(
        x = .25,
        y = 1,
        orientation='h'
    ),
    template='simple_white'
)

fig_pareto.add_hline(
    y=80, 
    line_dash='dot', 
    line_color='gray',
    line_width=2,
    yref='y2',
    annotation_text='80% Threshold',
    annotation_position='top right',
    annotation_font_color='gray'
)


fig_pareto.show()

# Customer Insights

In [77]:
# Customizable pie chart for revenue or units sold by segment/age group

# Note to self: make customizable in dashboard, but for now just do revenue by age group



df_cust_pie = df_net.groupby('customer_age_group')[['revenue_usd', 'units_sold']].sum().reset_index()

fig_pie_cust = px.pie(
    df_cust_pie,
    values = 'revenue_usd',
    names = 'customer_age_group',
    color_discrete_sequence=px.colors.qualitative.Bold,
    hole=0.3,
    template='simple_white',
    category_orders={'customer_age_group': ['18-24', '25-34', '35-44', '45-54', '55+']}
)

fig_pie_cust.update_layout(
    font_family='rockwell',
    title={
        'text': 'Revenue by Customer Age Group (2022-2025) <br> <sup style="font-size:12px;color:gray">Hover over slices to see specific USD values</sup>',
        'x': 0.5
    },
    legend_traceorder='reversed',
)

fig_pie_cust.update_traces(
    textposition='inside',
    textinfo='label+percent',
    hoverinfo='value',
    hovertemplate='$%{value:,.2f} USD<extra></extra>',
    marker=dict(line=dict(color='white', width=4))
)

fig_pie_cust.show()

In [75]:
# Chart for rating distribution

fig_hist = px.histogram(
    df_net,
    x = 'customer_rating',
    range_x = [1, 5.1],
    marginal='violin',
    opacity=0.5,
    labels={'customer_rating':'Customer Rating'},
    template='simple_white',
    color_discrete_sequence=['steelblue']
)

fig_hist.update_layout(
    font_family='rockwell',
    title={
        'text': 'Distribution of Customer Ratings (2022-2025) <br> <sup style="font-size:12px;color:gray">Hover over bars to see specific unit values</sup>',
        'x': 0.5
    },
    xaxis_title='Customer Rating',
    yaxis_title='Number of ratings',
)

fig_hist.update_traces(
    hovertemplate='<b>Customer Rating:</b> %{x}<br><b>Number of Ratings:</b> %{y}<extra></extra>'
)

fig_hist.show()

In [86]:
# Graph for return rate by segment/age group

df_return = (
    df
    .assign(is_returned=lambda x: x["return_status"] == "Returned")
    .groupby("customer_segment")["is_returned"]
    .mean()
    .reset_index(name="return_rate")
)

df_return['return_rate'] = round(df_return['return_rate'] * 100)

fig_return = go.Figure(data=[go.Table(
    header = dict(values = ['Customer Segment', 'Return Rate']),
    cells = dict(values = [df_return['customer_segment'], df_return['return_rate']])
)])

fig_return.update_layout(
    font_family='rockwell',
    title={
        "text": "Return Rate by Customer Segment",
        "x": 0.5
        }
)

fig_return.show()